In [1]:
# Table S1: the full 72-gene chaperone/protease census, with the inclusion rule
# and a flag for whether each gene also appears in the 61-gene high-confidence
# ATFS-1 regulon, plus ATFS-1 binding status for the two census genes that overlap
# the regulon and three additional permissive-only genes considered for census
# membership and checked but excluded on domain grounds (2026-09-23/2026-09-26
# decision to fold Table 1's content in here rather than maintain a separate,
# uncited "Table 1" - see the merge cells below). The census itself is frozen
# (data/chaperone_protease_census.csv) and was already independently reproduced
# from the raw annotation in census_build.ipynb; this notebook re-validates that
# reproduction here rather than trusting the frozen file blindly, then formats it
# as a supplementary table.
import os
if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

import gzip
import pandas as pd

FROZEN = "data/chaperone_protease_census.csv"
census = pd.read_csv(FROZEN)
if len(census) != 72:
    raise RuntimeError(f"Census has {len(census)} rows, expected 72.")
role_counts = census["role"].value_counts()
if role_counts.get("chaperone", 0) != 65 or role_counts.get("protease", 0) != 7:
    raise RuntimeError(f"Role split changed: {role_counts.to_dict()}, expected 65 chaperone / 7 protease.")
print(f"Census loaded: {len(census)} genes ({role_counts['chaperone']} chaperone, {role_counts['protease']} protease)")

Census loaded: 72 genes (65 chaperone, 7 protease)


In [2]:
# Independent reproduction check - the same rebuild census_build.ipynb performs,
# repeated here rather than assumed still true, since Table S1 is what a reviewer
# would actually check numbers against.
GFF = "data/raw/c_elegans.PRJNA13758.WS285.protein_annotation.gff3.gz"

SINGLE_DOMAIN = {
    "PF00012": "HSP70",       "PF00226": "DnaJ",
    "PF00183": "HSP90",       "PF00011": "HSP20",
    "PF00118": "Cpn60_TCP1",  "PF01920": "Prefoldin",
    "PF00574": "CLP_protease", "PF01434": "Peptidase_M41",
}
LON_DOMAINS = {"PF05362": "Lon_C", "PF02190": "LON_substr_bdg"}
PROTEASE_DOMAINS = {"PF00574", "PF01434", "PF05362", "PF02190"}
ALL_DOMAINS = set(SINGLE_DOMAIN) | set(LON_DOMAINS)

protein_gene, protein_domains, has_signal_peptide = {}, {}, set()
with gzip.open(GFF, "rt") as fh:
    for line in fh:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9:
            continue
        protein, source, feature, attrs = fields[0], fields[1], fields[2], fields[8]
        if source == "WormBase" and feature == "CDS":
            a = dict(kv.split("=", 1) for kv in attrs.split(";") if "=" in kv)
            if "wormbase_geneid" in a:
                protein_gene[protein] = (a["wormbase_geneid"], a.get("wormbase_genename", ""))
        elif source == "Pfam" and feature == "motif":
            for pfam in ALL_DOMAINS:
                if pfam in attrs:
                    protein_domains.setdefault(protein, set()).add(pfam)
        elif source == "SignalP" and feature == "signal_peptide":
            has_signal_peptide.add(protein)

qualifying, er_targeted = {}, set()
for protein, domains in protein_domains.items():
    if protein not in protein_gene:
        continue
    matched = domains & set(SINGLE_DOMAIN)
    if set(LON_DOMAINS) <= domains:
        matched |= set(LON_DOMAINS)
    if not matched:
        continue
    gene_id, gene_name = protein_gene[protein]
    if protein in has_signal_peptide:
        er_targeted.add(gene_id)
    entry = qualifying.setdefault(gene_id, {"name": gene_name, "domains": set()})
    entry["domains"] |= matched

rebuilt_census = {g: e for g, e in qualifying.items() if g not in er_targeted}
MANUAL_EXCLUSIONS = {"ppk-3", "lido-17", "rme-8"}
rebuilt_census = {g: e for g, e in rebuilt_census.items() if e["name"] not in MANUAL_EXCLUSIONS}

name_to_gene = {}
for protein, (gene_id, gene_name) in protein_gene.items():
    if gene_name:
        name_to_gene.setdefault(gene_name, gene_id)
for subunit in ("pfd-3", "pfd-5"):
    gene_id = name_to_gene[subunit]
    rebuilt_census[gene_id] = {"name": subunit, "domains": {"PF01920"}}

if set(rebuilt_census) != set(census["gene_id"]):
    raise RuntimeError("Rebuilt census gene membership no longer matches the frozen file.")
print(f"Reproduction check: {len(rebuilt_census)} genes, exact membership match against the frozen file.")

Reproduction check: 72 genes, exact membership match against the frozen file.


In [3]:
# Cross-reference: which of the 72 census genes also appear in the 61-gene
# high-confidence regulon (already computed and validated in table_s2.ipynb).
regulon = pd.read_csv("results/regulon_61.csv")
regulon_names = set(regulon["public_name"]) | set(regulon["seqname"])
census["in_61_regulon"] = census["public_name"].isin(regulon_names) | census["seqname"].isin(regulon_names)

n_in_regulon = int(census["in_61_regulon"].sum())
in_regulon_names = sorted(census.loc[census["in_61_regulon"], "public_name"])
if n_in_regulon != 2 or in_regulon_names != ["dnj-10", "ymel-1"]:
    raise RuntimeError(f"Expected exactly dnj-10 and ymel-1 in the regulon, got {in_regulon_names}.")
print(f"{n_in_regulon} of 72 census genes are in the 61-gene regulon: {in_regulon_names}")
print("(matches Analysis B's strict count exactly - this is the same fact from the other direction.)")

2 of 72 census genes are in the 61-gene regulon: ['dnj-10', 'ymel-1']
(matches Analysis B's strict count exactly - this is the same fact from the other direction.)


In [4]:
# Merge in ATFS-1 binding status for the two census genes that overlap the
# 61-gene regulon, per the 2026-09-23 decision to fold this into Table S1 rather
# than maintain a separate, uncited "Table 1". Sourced from table_1.ipynb's own
# already-validated output (results/table_1.csv), not recomputed here - same
# discipline as the in_61_regulon cross-reference above. Populated only for
# dnj-10 and ymel-1; binding was never assessed for the other 70 genes, so blank
# (NaN) is the correct value there, not a gap to fill.
table1_binding = pd.read_csv("results/table_1.csv").set_index("public_name")
BINDING_COLS = ["bound_published_soo2021", "bound_nargund2015", "bound_this_study"]
for col in BINDING_COLS:
    census[col] = census["public_name"].map(table1_binding[col])

n_populated = int(census["bound_published_soo2021"].notna().sum())
populated_names = sorted(census.loc[census["bound_published_soo2021"].notna(), "public_name"])
if n_populated != 2 or populated_names != ["dnj-10", "ymel-1"]:
    raise RuntimeError(f"Expected binding data on exactly dnj-10 and ymel-1, got {populated_names}.")

# The specific fact Results 2's closing sentence depends on: ymel-1 shows bound
# under Nargund 2015 even though Nargund's own table lists it as "yme-1", not
# ymel-1 - the alias a naive name-only lookup would miss entirely.
ymel1_nargund = census.loc[census["public_name"] == "ymel-1", "bound_nargund2015"].iloc[0]
if ymel1_nargund != "Yes":
    raise RuntimeError(f"ymel-1 Nargund 2015 binding = {ymel1_nargund!r}, expected 'Yes' - alias check failed.")

print(f"Binding columns merged: populated for {populated_names}, blank for the other {72 - n_populated} genes.")
print(f"ymel-1 Nargund 2015 binding (alias-sensitive check): {ymel1_nargund}")
print(census.loc[census['public_name'].isin(['dnj-10', 'ymel-1']),
                  ['public_name'] + BINDING_COLS].to_string(index=False))

Binding columns merged: populated for ['dnj-10', 'ymel-1'], blank for the other 70 genes.
ymel-1 Nargund 2015 binding (alias-sensitive check): Yes
public_name bound_published_soo2021 bound_nargund2015 bound_this_study
     dnj-10                      No                No               No
     ymel-1                      No               Yes              Yes


In [5]:
# Complete the 2026-09-23 decision: the merge above only covered the two genes
# already present as census rows (dnj-10, ymel-1). table_1.ipynb's own scope is
# five genes - those two plus three permissive-only genes considered for census
# membership and checked, but excluded on domain grounds (prx-19, cbp-3, tspo-1).
# Those three are not census members and are appended as three clearly-flagged
# extra rows rather than folded into the 72, so the table's own stated "72-gene
# census" scope stays accurate - the frozen census file itself is untouched.
# Sourced entirely from table_1.ipynb's already-validated output
# (results/table_1.csv, still loaded above as table1_binding) and
# results/regulon_61.csv (loaded above as regulon) for identity fields - neither
# is recomputed here.
BORDERLINE_GENES = ["prx-19", "cbp-3", "tspo-1"]
regulon_lookup = regulon.set_index("public_name")

borderline_rows = pd.DataFrame([
    {
        "gene_id": regulon_lookup.loc[gene, "wbgene"],
        "seqname": regulon_lookup.loc[gene, "seqname"],
        "public_name": gene,
        "role": "excluded",
        "pfam_families": table1_binding.loc[gene, "pfam_domain"],
        "in_61_regulon": True,
        "bound_published_soo2021": table1_binding.loc[gene, "bound_published_soo2021"],
        "bound_nargund2015": table1_binding.loc[gene, "bound_nargund2015"],
        "bound_this_study": table1_binding.loc[gene, "bound_this_study"],
    }
    for gene in BORDERLINE_GENES
])

if len(borderline_rows) != 3 or borderline_rows[BINDING_COLS].isna().any().any():
    raise RuntimeError("Borderline-gene rows incomplete - check table_1.csv coverage.")

census = pd.concat([census, borderline_rows], ignore_index=True)
if len(census) != 75:
    raise RuntimeError(f"Expected 75 rows after appending the 3 borderline genes, got {len(census)}.")

print(f"Appended {len(borderline_rows)} borderline rows (not census members, role='excluded'): "
      f"{BORDERLINE_GENES}")
print(f"Table now covers {len(census)} rows: 72 census genes + 3 borderline genes with binding data.")
print(f"Genes with full binding coverage: {int(census['bound_published_soo2021'].notna().sum())} of {len(census)}.")

Appended 3 borderline rows (not census members, role='excluded'): ['prx-19', 'cbp-3', 'tspo-1']
Table now covers 75 rows: 72 census genes + 3 borderline genes with binding data.
Genes with full binding coverage: 5 of 75.


In [6]:
# Assemble and write. Column order: identity, role, domain evidence, inclusion
# note (manual additions carry their own explanatory text, everything else is
# domain-only), regulon membership flag, then the three binding columns (blank
# except for the five genes with binding data, per the 2026-09-23 decision to
# fold Table 1 into Table S1 rather than maintain a separate, uncited "Table 1").
table_s1 = census.rename(columns={"gene_id": "wbgene"})[
    ["wbgene", "seqname", "public_name", "role", "pfam_families", "in_61_regulon",
     "bound_published_soo2021", "bound_nargund2015", "bound_this_study"]
].copy()
# Sorted for the reader: chaperones, then proteases (the majority-first framing
# used throughout the paper), then the three excluded-but-checked borderline
# genes as their own trailing group, gene name in natural order within each
# group. A plain string sort puts "dnj-10" before "dnj-2" - fine for a computer,
# not for anyone reading the table - so digit runs are zero-padded before sorting
# and the padded key is dropped again afterward.
import re
def _natural_key(s):
    return re.sub(r"\d+", lambda m: m.group().zfill(10), str(s).lower())
ROLE_ORDER = {"chaperone": 0, "protease": 1, "excluded": 2}
table_s1["_role_order"] = table_s1["role"].map(ROLE_ORDER)
table_s1["_namekey"] = table_s1["public_name"].map(_natural_key)
table_s1 = table_s1.sort_values(["_role_order", "_namekey"]).drop(columns=["_role_order", "_namekey"]).reset_index(drop=True)

INCLUSION_RULE = (
    "Inclusion rule: membership decided from Pfam protein-domain annotations, not gene "
    "name. Eight domain families qualify a gene on their own (HSP70, HSP20, HSP90, DnaJ, "
    "Cpn60_TCP1, Prefoldin, CLP_protease, Peptidase_M41); the Lon protease additionally "
    "requires both Lon_C and LON_substr_bdg together, since either alone is not sufficient. "
    "Genes with a signal peptide co-occurring with a qualifying domain on the same protein "
    "isoform are excluded as ER-targeted. Two manual exclusions (ppk-3, rme-8) remove genes "
    "where the matched domain is a minor accessory feature rather than the protein's "
    "function. Two manual additions (pfd-3, pfd-5) restore canonical prefoldin subunits that "
    "carry no Pfam hit in this WormBase release but are unambiguous complex members. The "
    "final three rows (role = 'excluded') are not census members: they are permissive-only "
    "genes considered for inclusion and checked, but excluded because they carry no "
    "qualifying domain, kept here because ATFS-1 binding status was assessed for them "
    "alongside the two genuine census members that overlap the 61-gene regulon. ATFS-1 "
    "binding status (published, Nargund 2015, and this study's own reconstruction) is given "
    "for five genes: dnj-10 and ymel-1, the two census genes that also fall in the 61-gene "
    "high-confidence regulon, plus prx-19, cbp-3, and tspo-1, the three excluded genes just "
    "described; binding was not assessed for the other 70 genuine census genes."
)

os.makedirs("tables", exist_ok=True)
table_s1.to_csv("results/table_s1_census.csv", index=False)
table_s1.to_csv("tables/table_s1_census.csv", index=False)
with open("tables/table_s1_census_note.txt", "w") as fh:
    fh.write(INCLUSION_RULE + "\n")

print(f"Wrote {len(table_s1)} rows to results/table_s1_census.csv and tables/table_s1_census.csv")
print(f"Wrote inclusion-rule note to tables/table_s1_census_note.txt")
print(f"\n{table_s1['role'].value_counts().to_dict()}, "
      f"{int(table_s1['in_61_regulon'].sum())} in the 61-gene regulon, "
      f"{int(table_s1['bound_published_soo2021'].notna().sum())} with binding data")
print(table_s1.columns.tolist())
print(table_s1.tail(6).to_string(index=False))

Wrote 75 rows to results/table_s1_census.csv and tables/table_s1_census.csv
Wrote inclusion-rule note to tables/table_s1_census_note.txt

{'chaperone': 65, 'protease': 7, 'excluded': 3}, 5 in the 61-gene regulon, 5 with binding data
['wbgene', 'seqname', 'public_name', 'role', 'pfam_families', 'in_61_regulon', 'bound_published_soo2021', 'bound_nargund2015', 'bound_this_study']
        wbgene   seqname public_name     role      pfam_families  in_61_regulon bound_published_soo2021 bound_nargund2015 bound_this_study
WBGene00004978 Y47G6A.10       spg-7 protease      Peptidase_M41          False                     NaN               NaN              NaN
WBGene00021615  Y47C4A.1    Y47C4A.1 protease      Peptidase_M41          False                     NaN               NaN              NaN
WBGene00010842  M03C11.5      ymel-1 protease      Peptidase_M41           True                      No               Yes              Yes
WBGene00009595  F40F12.7       cbp-3 excluded   PF02135 (zf-TAZ)

In [7]:
# Render the manuscript-ready PDF, same journal spec and font as the figures.
#
# Landscape orientation (page_w/page_h swapped), not the default portrait every
# other table here uses: measured against the real renderer (table_style._measurer)
# rather than guessed, the 9 columns' bare-minimum widths summed to 0.9987 of
# portrait's usable width - essentially zero margin, and a first attempt at
# portrait-width columns already overflowed once (both in the body, from
# single-token values like WBGene IDs and "chaperone" that table_style's wrapper
# can never break since it only wraps on whitespace, and again in the header row,
# from tokens like "(published)" and "Nargund" at bold header weight). Landscape
# gives ~19% more usable width, enough for a real safety margin on every column
# instead of a fit with no room for rendering variance. Header labels for the
# three binding columns are also shortened ("Soo 2021" / "Nargund 2015" / "This
# study" rather than "Bound (...)") since the parenthetical form is itself a
# single unbreakable token wider than the column needs to be for its "Yes"/"No"
# body values.
import sys
sys.path.insert(0, "scripts")
from table_style import render_table
from figure_style import FULL_W, MAX_H

COLUMNS = [
    {"key": "wbgene", "label": "WBGene ID", "width": 0.155, "wrap": True},
    {"key": "seqname", "label": "Sequence name", "width": 0.105, "wrap": True},
    {"key": "public_name", "label": "Gene", "width": 0.09, "wrap": True},
    {"key": "role", "label": "Role", "width": 0.10, "wrap": True},
    {"key": "pfam_families", "label": "Pfam domain(s)", "width": 0.22, "wrap": True},
    {"key": "in_61_regulon", "label": "In 61-gene regulon?", "width": 0.095,
     "align": "center", "wrap": True, "format": lambda v: "Yes" if v else ""},
    {"key": "bound_published_soo2021", "label": "Bound: Soo 2021", "width": 0.065,
     "align": "center", "wrap": True, "format": lambda v: v if pd.notna(v) else ""},
    {"key": "bound_nargund2015", "label": "Bound: Nargund 2015", "width": 0.095,
     "align": "center", "wrap": True, "format": lambda v: v if pd.notna(v) else ""},
    {"key": "bound_this_study", "label": "Bound: this study", "width": 0.075,
     "align": "center", "wrap": True, "format": lambda v: v if pd.notna(v) else ""},
]
n_pages = render_table(
    table_s1, COLUMNS,
    title=("Table S1. The full 72-gene chaperone/protease census, plus ATFS-1 binding "
           "status for three additional borderline genes considered but excluded."),
    filename="table_s1_census",
    footnote=INCLUSION_RULE,
    page_w=MAX_H, page_h=FULL_W,
)
print(f"Table S1 rendered to {n_pages} page(s).")

Wrote tables/table_s1_census.pdf (4 page(s)); preview PNG(s): ['tables/table_s1_census_page1.png', 'tables/table_s1_census_page2.png', 'tables/table_s1_census_page3.png', 'tables/table_s1_census_page4.png']
Table S1 rendered to 4 page(s).
